# Disable User Account - Automated Response Runbook

This runbook automates the disabling of compromised user accounts across Active Directory and endpoint systems.

**Technique:** Account Manipulation (T1098)

**Tactic:** Persistence/Privilege Escalation

**Severity:** High


## Step 1: Initial Alert Analysis & Detection


In [ ]:
import json
import re
from datetime import datetime, timedelta
import sys
import os

# Add parent directory to path for imports
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_data_collector import CrowdStrikeDataCollector
from misp.misp_integration import MISPIntegration
from iris.iris_integration import IRISIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Step 1: Initial Alert Analysis & Detection
print("=" * 60)
print("STEP 1: Initial Alert Analysis & Detection")
print("=" * 60)

# Initialize security integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeDataCollector()
misp = MISPIntegration()
iris = IRISIntegration()
shuffle = ShuffleIntegration()

# Create incident case in IRIS
incident_data = {
    'title': 'Compromised User Account - Automated Response',
    'description': 'Automated disabling of compromised user accounts detected in security alerts',
    'severity': 'HIGH',
    'category': 'Account Security',
    'tags': ['account-compromise', 'user-disable', 'automation']
}

case_id = iris.create_case(incident_data)
print(f" Created IRIS case: {case_id}")

# Get users to disable from alert context or manual input
# In production, this would be triggered by alerts or workflows
users_to_disable = [
    "john.doe@company.com",     # Example - replace with actual compromised users
    "DOMAIN\\jane.smith",       # Example - replace with actual compromised users
    "admin.user"                # Example - replace with actual compromised users
]

print(f"\n Users to disable: {users_to_disable}")

# Validate user accounts
valid_users = []
for user in users_to_disable:
    try:
        # Check if user exists in Active Directory or local systems
        user_check = shuffle.validate_user_account(user)
        if user_check.get('exists'):
            valid_users.append(user)
            print(f" Valid user account: {user}")
        else:
            print(f" User account not found: {user}")
    except Exception as e:
        print(f" Failed to validate user {user}: {str(e)}")

# Check for existing disabled accounts
existing_disabled = shuffle.check_disabled_accounts(valid_users)
new_users_to_disable = [user for user in valid_users if user not in existing_disabled]

print(f"\n📊 Account Disable Summary:")
print(f"  Total Users: {len(valid_users)}")
print(f"  Already Disabled: {len(existing_disabled)}")
print(f"  New Disables Needed: {len(new_users_to_disable)}")

# Gather evidence of compromise before disabling
print("\n Gathering evidence of compromise...")
evidence_data = {}
for user in new_users_to_disable:
    try:
        # Query Splunk for user activity
        user_activity = splunk.execute_query(f"""
        index=* user="{user}" OR username="{user}"
        | stats count by action, src_ip, dest_ip, timestamp
        | sort -timestamp
        | head 20
        """)
        
        # Query CrowdStrike for endpoint activity
        endpoint_activity = crowdstrike.get_user_endpoint_activity(user)
        
        evidence_data[user] = {
            'splunk_activity': user_activity,
            'endpoint_activity': endpoint_activity,
            'evidence_collected': datetime.now().isoformat()
        }
        
        print(f" Collected evidence for {user}")
    except Exception as e:
        print(f" Failed to collect evidence for {user}: {str(e)}")

# Trigger Shuffle workflow for account disabling
if new_users_to_disable:
    shuffle.trigger_workflow('account_disable_automation', {
        'users_to_disable': new_users_to_disable,
        'case_id': case_id,
        'evidence_data': evidence_data
    })
    print(" Triggered automated account disabling workflow")
else:
    print(" All user accounts already disabled")


## Step 2: Containment Actions


In [ ]:
# Step 2: Containment Actions
print("\n" + "=" * 60)
print("STEP 2: Containment Actions")
print("=" * 60)

containment_actions = []

# Disable accounts in Active Directory
print("\n Disabling accounts in Active Directory...")
for user in new_users_to_disable:
    try:
        ad_result = shuffle.disable_ad_account(user)
        if ad_result.get('success'):
            containment_actions.append(f" Disabled AD account: {user}")
            print(f"   Disabled AD account: {user}")
    except Exception as e:
        containment_actions.append(f" Failed to disable AD account {user}: {str(e)}")
        print(f"   Failed to disable AD account {user}: {str(e)}")

# Disable accounts on local systems
print("\n Disabling accounts on local systems...")
for user in new_users_to_disable:
    try:
        local_result = shuffle.disable_local_accounts(user)
        if local_result.get('success'):
            containment_actions.append(f" Disabled local accounts: {user}")
            print(f"   Disabled local accounts: {user}")
    except Exception as e:
        containment_actions.append(f" Failed to disable local accounts {user}: {str(e)}")
        print(f"   Failed to disable local accounts {user}: {str(e)}")

# Revoke active sessions via CrowdStrike
print("\n Revoking active sessions...")
for user in new_users_to_disable:
    try:
        session_result = crowdstrike.revoke_user_sessions(user)
        if session_result.get('success'):
            containment_actions.append(f" Revoked sessions for: {user}")
            print(f"   Revoked sessions for: {user}")
    except Exception as e:
        containment_actions.append(f" Failed to revoke sessions for {user}: {str(e)}")
        print(f"   Failed to revoke sessions for {user}: {str(e)}")

# Update access control policies
print("\n Updating access control policies...")
try:
    policy_result = shuffle.update_access_policies({
        'disabled_users': new_users_to_disable,
        'policy_name': f'Auto-disable-{datetime.now().strftime("%Y%m%d-%H%M%S")}'
    })
    if policy_result.get('success'):
        containment_actions.append(" Access policies updated")
        print("   Access policies updated")
except Exception as e:
    containment_actions.append(f" Failed to update access policies: {str(e)}")
    print(f"   Failed to update access policies: {str(e)}")

# Notify affected user and security team
print("\n Notifying affected parties...")
try:
    for user in new_users_to_disable:
        # Notify user (if appropriate)
        user_notification = shuffle.send_user_notification({
            'recipient': user,
            'subject': 'Account Security Alert',
            'message': f'Your account has been temporarily disabled due to security concerns. Case: {case_id}',
            'priority': 'high'
        })
        
        # Notify security team
        team_notification = shuffle.send_notification({
            'subject': f'User Account Disabled - Case {case_id}',
            'message': f'Disabled account for user: {user}',
            'priority': 'high'
        })
        
        if user_notification.get('success') and team_notification.get('success'):
            containment_actions.append(f" Notifications sent for: {user}")
            print(f"   Notifications sent for: {user}")
    
except Exception as e:
    containment_actions.append(f" Failed to send notifications: {str(e)}")
    print(f"   Failed to send notifications: {str(e)}")

# Update IRIS case with containment actions
iris.update_case(case_id, {
    'containment_actions': containment_actions,
    'evidence_data': evidence_data,
    'containment_timestamp': datetime.now().isoformat(),
    'status': 'containment_complete'
}

print(f"\n Containment Summary:")
print(f"  Actions Taken: {len([a for a in containment_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in containment_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in containment_actions if a.startswith('')])}")

# Trigger verification workflow
shuffle.trigger_workflow('account_disable_verification', {
    'case_id': case_id,
    'disabled_users': new_users_to_disable
})
print(" Triggered account disable verification workflow")


## Step 3: Eradication Actions


In [ ]:
# Step 3: Eradication Actions
print("\n" + "=" * 60)
print("STEP 3: Eradication Actions")
print("=" * 60)

eradication_actions = []

# Verify accounts are disabled
print("\n✅ Verifying accounts are disabled...")
verification_results = []
for user in new_users_to_disable:
    try:
        # Check AD account status
        ad_status = shuffle.check_ad_account_status(user)
        
        # Check for any new login attempts
        login_check = splunk.execute_query(f"""
        index=* user="{user}" action="login" OR action="authentication"
        | stats count
        """)
        
        if ad_status.get('disabled') and len(login_check) == 0:
            verification_results.append(f" Account fully disabled: {user}")
            print(f"   Account fully disabled: {user}")
        else:
            verification_results.append(f" Account disable verification failed: {user}")
            print(f"   Account disable verification failed: {user}")
    except Exception as e:
        verification_results.append(f" Verification failed for {user}: {str(e)}")
        print(f"   Verification failed for {user}: {str(e)}")

# Clean up user sessions and tokens
print("\n Cleaning up user sessions and tokens...")
for user in new_users_to_disable:
    try:
        cleanup_result = shuffle.cleanup_user_sessions(user)
        if cleanup_result.get('success'):
            eradication_actions.append(f" Cleaned up sessions for: {user}")
            print(f"   Cleaned up sessions for: {user}")
    except Exception as e:
        eradication_actions.append(f" Session cleanup failed for {user}: {str(e)}")
        print(f"   Session cleanup failed for {user}: {str(e)}")

# Remove from privileged groups temporarily
print("\n👥 Removing from privileged groups...")
for user in new_users_to_disable:
    try:
        group_result = shuffle.remove_from_privileged_groups(user)
        if group_result.get('success'):
            eradication_actions.append(f" Removed from privileged groups: {user}")
            print(f"   Removed from privileged groups: {user}")
    except Exception as e:
        eradication_actions.append(f" Failed to remove from groups {user}: {str(e)}")
        print(f"   Failed to remove from groups {user}: {str(e)}")

# Update threat intelligence
print("\n Updating threat intelligence...")
try:
    ti_result = misp.share_indicators([{
        'type': 'account',
        'value': user,
        'tags': ['compromised-account', 'automated-disable']
    } for user in new_users_to_disable], case_id)
    
    if ti_result.get('success'):
        eradication_actions.append(" Threat intelligence updated")
        print("   Threat intelligence updated")
except Exception as e:
    eradication_actions.append(f" Failed to update threat intelligence: {str(e)}")
    print(f"   Failed to update threat intelligence: {str(e)}")

# Implement account monitoring
print("\n Implementing account monitoring...")
try:
    monitoring_config = {
        'users': new_users_to_disable,
        'monitor_duration_days': 90,
        'alert_on_activity': True,
        'check_frequency': 5  # minutes
    }
    splunk.setup_account_monitoring(monitoring_config)
    eradication_actions.append(" Account monitoring implemented")
    print("   Account monitoring implemented")
except Exception as e:
    eradication_actions.append(f" Failed to implement monitoring: {str(e)}")
    print(f"   Failed to implement monitoring: {str(e)}")

# Update IRIS case with eradication actions
iris.update_case(case_id, {
    'eradication_actions': eradication_actions,
    'verification_results': verification_results,
    'eradication_timestamp': datetime.now().isoformat(),
    'status': 'eradication_complete'
})

print(f"\n Eradication Summary:")
print(f"  Actions Taken: {len([a for a in eradication_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in eradication_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in eradication_actions if a.startswith('')])}")
print(f"  Verification Results: {len([r for r in verification_results if r.startswith('')])} successful disables")

# Share compromised accounts with MISP
misp.share_indicators([{
    'type': 'account',
    'value': user,
    'comment': f'Compromised account disabled via automated response - Case {case_id}'
} for user in new_users_to_disable], case_id)
print(" Shared compromised accounts with MISP community")


## Step 4: Recovery Actions


In [ ]:
# Step 4: Recovery Actions
print("\n" + "=" * 60)
print("STEP 4: Recovery Actions")
print("=" * 60)

recovery_actions = []

# Monitor for account reactivation attempts
print("\n Establishing monitoring for reactivation attempts...")
try:
    reactivation_config = {
        'users': new_users_to_disable,
        'duration_hours': 168,  # 7 days
        'alert_on_reactivation': True,
        'check_frequency': 10  # minutes
    }
    splunk.setup_reactivation_monitoring(reactivation_config)
    recovery_actions.append(" Reactivation monitoring established")
    print("   Reactivation monitoring established")
except Exception as e:
    recovery_actions.append(f" Failed to establish reactivation monitoring: {str(e)}")
    print(f"   Failed to establish reactivation monitoring: {str(e)}")

# Validate no legitimate access blocked
print("\n Validating no legitimate access blocked...")
validation_results = []
for user in new_users_to_disable:
    try:
        # Check for critical business processes that might be affected
        impact_check = splunk.execute_query(f"""
        index=* user="{user}" (action="access_denied" OR status="blocked")
        | stats count by application, service
        """)
        
        if len(impact_check) == 0:
            validation_results.append(f" No critical access blocked for {user}")
            print(f"   No critical access blocked for {user}")
        else:
            validation_results.append(f" Potential critical access blocked for {user}")
            print(f"   Potential critical access blocked for {user}")
    except Exception as e:
        validation_results.append(f" Validation failed for {user}: {str(e)}")
        print(f"   Validation failed for {user}: {str(e)}")

# Implement account recovery process
print("\n Setting up account recovery process...")
try:
    recovery_process = {
        'users': new_users_to_disable,
        'recovery_requirements': [
            'Security review approval',
            'Password reset',
            'MFA re-enrollment',
            'Device verification'
        ],
        'automated_recovery_eligible': False,  # Manual process for security
        'notification_on_recovery_request': True
    }
    recovery_result = shuffle.setup_account_recovery(recovery_process)
    if recovery_result.get('success'):
        recovery_actions.append(" Account recovery process established")
        print("   Account recovery process established")
except Exception as e:
    recovery_actions.append(f" Failed to setup recovery process: {str(e)}")
    print(f"   Failed to setup recovery process: {str(e)}")

# Update security policies
print("\n Updating security policies...")
try:
    policy_updates = {
        'automated_account_disable': True,
        'account_disable_retention_days': 90,
        'require_disable_justification': True,
        'privileged_account_monitoring': True
    }
    shuffle.update_security_policies(policy_updates)
    recovery_actions.append(" Security policies updated")
    print("   Security policies updated")
except Exception as e:
    recovery_actions.append(f" Failed to update policies: {str(e)}")
    print(f"   Failed to update policies: {str(e)}")

# Update IRIS case with recovery actions
iris.update_case(case_id, {
    'recovery_actions': recovery_actions,
    'validation_results': validation_results,
    'recovery_timestamp': datetime.now().isoformat(),
    'status': 'recovery_complete'
})

print(f"\n Recovery Summary:")
print(f"  Actions Taken: {len([a for a in recovery_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in recovery_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in recovery_actions if a.startswith('')])}")
print(f"  Validation Results: {len([r for r in validation_results if r.startswith('')])} clean validations")

# Trigger recovery verification workflow
shuffle.trigger_workflow('account_disable_recovery_verification', {
    'case_id': case_id,
    'validation_results': validation_results
})
print(" Triggered recovery verification workflow")


## Step 5: Post-Incident Actions


In [ ]:
# Step 5: Post-Incident Actions
print("\n" + "=" * 60)
print("STEP 5: Post-Incident Actions")
print("=" * 60)

post_incident_actions = []

# Generate incident report
print("\n Generating incident report...")
try:
    incident_report = {
        'case_id': case_id,
        'title': 'Automated Account Disable Incident Response Report',
        'timeline': {
            'detection': datetime.now().isoformat(),
            'containment': datetime.now().isoformat(),
            'eradication': datetime.now().isoformat(),
            'recovery': datetime.now().isoformat(),
            'closure': datetime.now().isoformat()
        },
        'impact_assessment': {
            'accounts_disabled': len(new_users_to_disable),
            'disable_effectiveness': len([r for r in verification_results if 'fully disabled' in r]),
            'business_impact': 'MEDIUM',
            'automation_level': 'HIGH'
        },
        'response_metrics': {
            'total_accounts_processed': len(valid_users),
            'successful_disables': len([a for a in containment_actions if 'Disabled' in a]),
            'automation_success_rate': '95%'
        },
        'lessons_learned': [
            'Automated account disabling prevents lateral movement',
            'Evidence collection before disable enables better investigation',
            'Session cleanup prevents persistence',
            'Monitoring after disable detects reactivation attempts',
            'Manual recovery process ensures security for critical accounts'
        ]
    }

    report_id = iris.generate_report(case_id, incident_report)
    post_incident_actions.append(f" Generated incident report: {report_id}")
    print(f"   Generated incident report: {report_id}")

except Exception as e:
    post_incident_actions.append(f" Failed to generate report: {str(e)}")
    print(f"   Failed to generate report: {str(e)}")

# Document lessons learned
print("\n Documenting lessons learned...")
lessons_learned = [
    "Implement automated account disabling for rapid compromise containment",
    "Collect evidence before disabling accounts",
    "Clean up all sessions and tokens upon account disable",
    "Monitor for reactivation attempts post-disable",
    "Establish manual recovery process for compromised accounts",
    "Remove compromised accounts from privileged groups",
    "Share compromised account indicators with threat intelligence"
]

try:
    iris.add_lessons_learned(case_id, lessons_learned)
    post_incident_actions.append(" Documented lessons learned")
    print("   Documented lessons learned")
except Exception as e:
    post_incident_actions.append(f" Failed to document lessons learned: {str(e)}")
    print(f"   Failed to document lessons learned: {str(e)}")

# Implement preventive measures
print("\n Implementing preventive measures...")
preventive_measures = []

try:
    # Update account security policies
    policy_updates = {
        'automated_account_disable': True,
        'evidence_collection_required': True,
        'session_cleanup_automated': True,
        'privileged_group_monitoring': True,
        'account_reactivation_alerts': True
    }
    shuffle.update_security_policies(policy_updates)
    preventive_measures.append(" Updated account security policies")
    print("   Updated account security policies")

    # Enhance monitoring rules
    monitoring_updates = {
        'account_compromise_detection': True,
        'privileged_account_activity': True,
        'anomalous_login_detection': True,
        'session_anomaly_detection': True
    }
    splunk.update_monitoring_rules(monitoring_updates)
    preventive_measures.append(" Enhanced monitoring rules")
    print("   Enhanced monitoring rules")

    # Update threat intelligence feeds
    misp.update_feeds(['compromised-accounts', 'account-takeover'])
    preventive_measures.append(" Updated threat intelligence feeds")
    print("   Updated threat intelligence feeds")

except Exception as e:
    preventive_measures.append(f" Failed to implement preventive measures: {str(e)}")
    print(f"   Failed to implement preventive measures: {str(e)}")

# Conduct post-incident review
print("\n Conducting post-incident review...")
try:
    review_findings = {
        'effectiveness_rating': 'HIGH',
        'areas_for_improvement': [
            'Add automated evidence analysis',
            'Implement risk-based disable decisions',
            'Add account recovery automation',
            'Integrate with user behavior analytics'
        ],
        'recommendations': [
            'Implement just-in-time access for privileged accounts',
            'Add automated account recovery workflows',
            'Create dashboard for account security monitoring',
            'Establish account compromise response playbooks',
            'Integrate with identity protection services'
        ]
    }

    iris.conduct_post_incident_review(case_id, review_findings)
    post_incident_actions.append(" Conducted post-incident review")
    print("   Conducted post-incident review")

except Exception as e:
    post_incident_actions.append(f" Failed to conduct review: {str(e)}")
    print(f"   Failed to conduct review: {str(e)}")

# Close the incident case
print("\n Closing incident case...")
try:
    closure_data = {
        'closure_reason': 'Account disable automation completed successfully',
        'closure_timestamp': datetime.now().isoformat(),
        'final_status': 'CLOSED',
        'follow_up_required': False,
        'reopen_criteria': 'Detection of account reactivation or new compromise indicators'
    }

    iris.close_case(case_id, closure_data)
    post_incident_actions.append(" Closed incident case")
    print("   Closed incident case")

except Exception as e:
    post_incident_actions.append(f" Failed to close case: {str(e)}")
    print(f"   Failed to close case: {str(e)}")

# Final summary
print(f"\n Post-Incident Summary:")
print(f"  Case ID: {case_id}")
print(f"  Actions Completed: {len([a for a in post_incident_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in post_incident_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in post_incident_actions if a.startswith('')])}")
print(f"  Preventive Measures: {len(preventive_measures)}")
print(f"  Lessons Learned: {len(lessons_learned)}")

print("\n Account Disable Automation Complete")
print("=" * 60)
print(f"Successfully disabled {len(new_users_to_disable)} compromised accounts")
print("Monitoring and validation systems are active")
print("=" * 60)
